# Data Preparation — Malware Detection

Nesta fase são aplicadas as operações necessárias para preparar os dados para o desenvolvimento dos modelos de Machine Learning.

As decisões tomadas nesta etapa resultam das conclusões obtidas durante o Data Understanding.

Serão realizadas operações de limpeza, remoção de características desnecessárias ou redundantes, análise de duplicados, separação entre features e target, divisão dos dados em treino e teste e standardização das características.

In [ ]:
import sys
import subprocess
from pathlib import Path

em_colab = "google.colab" in sys.modules

if em_colab:
    pasta_projeto = Path("/content/IAAC-Projeto")

    if not pasta_projeto.exists():
        subprocess.run(
            [
                "git",
                "clone",
                "https://github.com/diogocarvalho2025056-crypto/IAAC-Projeto.git"
            ],
            check=True
        )
else:
    pasta_projeto = Path.cwd().parent

if str(pasta_projeto) not in sys.path:
    sys.path.insert(0, str(pasta_projeto))

print("Pasta do projeto:")
print(pasta_projeto)

In [ ]:
# --------------------------------------------------
# 1. INSTALAÇÃO DE DEPENDENCIAS DO KAGGLE
# --------------------------------------------------

%pip install -q kagglehub[pandas-datasets]

In [ ]:
# --------------------------------------------------
# 2. IMPORTS
# --------------------------------------------------

import pandas as pd
import kagglehub

from kagglehub import KaggleDatasetAdapter

from src.data_preparation_utils import (
    carregar_dataset,
    remover_colunas,
    verificar_missing_values,
    verificar_duplicados,
    verificar_perfis_repetidos,
    verificar_conflitos_target,
    remover_duplicados,
    guardar_dataset,
    separar_features_target,
    dividir_treino_teste,
    distribuicao_classes,
    standardizar_dados
)

## Importação do dataset

O dataset é importado diretamente da plataforma Kaggle.

É utilizado o ficheiro combinado `Malware_and_benign_recognition.csv`, contendo exemplos de ficheiros benignos e maliciosos.

A preparação será efetuada sobre uma cópia do dataset original, preservando os dados importados sem alterações.

In [ ]:
# Import dataset from Kaggle

file_path = "Malware_and_benign_recognition.csv"

df = kagglehub.load_dataset(
    KaggleDatasetAdapter.PANDAS,
    "aashishzzz/malware-detection-dataset-with-new-malware-info",
    file_path
)

print("Dataset importado com sucesso.")
print("Dimensão:", df.shape)

## Criação de uma cópia de trabalho

Para preservar os dados originalmente importados, é criada uma cópia do DataFrame.

Todas as operações de preparação serão realizadas sobre `df_prep`.

In [ ]:
df_prep = df.copy()

print("Dataset original:", df.shape)
print("Cópia de preparação:", df_prep.shape)

## Remoção de características desnecessárias

Com base no Data Understanding foram identificadas três características a remover:

- `File` — funciona como identificador do ficheiro e não representa uma característica estrutural adequada para o treino;
- `SuspiciousSectionNames` — apresenta sempre o mesmo valor e, por isso, não possui capacidade discriminativa;
- `SectionCount` — contém a mesma informação que `NumberOfSections`, sendo redundante.

Será mantida `NumberOfSections`.

In [ ]:
colunas_remover = [
    "File",
    "SuspiciousSectionNames",
    "SectionCount"
]

df_prep = remover_colunas(
    df_prep,
    colunas_remover
)

print("Dimensão após remoção das colunas:")
print(df_prep.shape)

print("\nColunas restantes:")
print(df_prep.columns.tolist())

## Valores em falta

É verificada a existência de valores em falta após a remoção das características anteriormente identificadas.

A análise realizada na fase de Data Understanding não tinha identificado missing values, mas a verificação é novamente efetuada antes do treino.

In [ ]:
missing = verificar_missing_values(
    df_prep
)

print("Número total de valores em falta:")
print(missing)

## Análise de duplicados

No dataset original não tinham sido identificadas linhas completamente duplicadas.

No entanto, após a remoção da coluna `File`, diferentes ficheiros podem passar a apresentar exatamente as mesmas características estruturais.

Por esse motivo, os duplicados são novamente analisados nesta fase.

In [ ]:
duplicados = verificar_duplicados(
    df_prep
)

print("Duplicados completos:")
print(duplicados)

In [ ]:
# Perfis repetidos

perfis_repetidos = verificar_perfis_repetidos(
    df_prep,
    "Malicious"
)

print("Número de registos com features repetidas:")
print(len(perfis_repetidos))

display(perfis_repetidos)

### Verificação dos rótulos

Antes de remover os duplicados é necessário verificar se existem exemplos com exatamente as mesmas features, mas classificados com rótulos diferentes.

Uma situação desse tipo representaria uma inconsistência relevante no dataset.

In [ ]:
conflitos = verificar_conflitos_target(
    df_prep,
    "Malicious"
)

print("Número de perfis com conflito:")
print(len(conflitos))

display(conflitos)

### Remoção dos duplicados

Como não foram identificados rótulos contraditórios entre os perfis repetidos, os duplicados completos podem ser removidos.

Esta operação evita que exemplos estruturalmente idênticos sejam representados mais do que uma vez no conjunto de dados.

In [ ]:
df_prep = remover_duplicados(
    df_prep
)

print("Dimensão após remoção dos duplicados:")
print(df_prep.shape)

print(
    "Duplicados restantes:",
    verificar_duplicados(df_prep)
)

## Dataset preparado - Guardar Dataset preparado

Após a limpeza e preparação inicial, é gerado um novo ficheiro `Malware_prepared.csv`.

Desta forma, o dataset originalmente obtido através do Kaggle é preservado, enquanto as fases seguintes do projeto podem utilizar uma versão consistente dos dados preparados.

O ficheiro contém 1.148 registos e 24 colunas, incluindo a variável alvo `Malicious`.

In [ ]:
ficheiro_preparado = (
    pasta_projeto
    / "dataset"
    / "Malware_prepared.csv"
)

guardar_dataset(
    df_prep,
    ficheiro_preparado
)

print("Dataset preparado guardado em:")
print(ficheiro_preparado)

## Separação entre features e target

A variável `Malicious` corresponde à variável alvo do problema:

- `0` — ficheiro benigno;
- `1` — ficheiro malicioso.

As restantes 23 colunas serão utilizadas como features para o desenvolvimento dos modelos.

In [ ]:
X, y = separar_features_target(
    df_prep,
    "Malicious"
)

print("Features:")
print(X.shape)

print("\nTarget:")
print(y.shape)

## Divisão em treino e teste

Os dados são divididos em:

- 80% para treino;
- 20% para teste.

É utilizado `random_state=42` para garantir reprodutibilidade.

É também utilizado `stratify=y`, permitindo preservar aproximadamente a distribuição das classes nos conjuntos de treino e teste.

In [ ]:
X_train, X_test, y_train, y_test = dividir_treino_teste(
    X,
    y
)

print("X_train:", X_train.shape)
print("X_test:", X_test.shape)
print("y_train:", y_train.shape)
print("y_test:", y_test.shape)

In [ ]:
# Distribuição das classes

print("--- DISTRIBUIÇÃO NO TREINO ---")
display(
    distribuicao_classes(y_train)
)

print("\n--- DISTRIBUIÇÃO NO TESTE ---")
display(
    distribuicao_classes(y_test)
)

### Distribuição após a divisão

A utilização de `stratify=y` permitiu manter uma distribuição praticamente equilibrada entre ficheiros benignos e maliciosos nos conjuntos de treino e teste.

Não se justifica, portanto, a aplicação de técnicas de oversampling, undersampling ou SMOTE nesta fase.

## Feature Scaling

As características apresentam escalas muito diferentes.

Por exemplo, algumas variáveis representam entropia, enquanto outras representam tamanhos de ficheiros ou endereços com valores muito superiores.

É aplicada standardização através do `StandardScaler`.

O scaler é ajustado exclusivamente com os dados de treino. Posteriormente, a transformação aprendida é aplicada aos dados de teste.

Esta abordagem evita data leakage.

In [ ]:
X_train_scaled, X_test_scaled, scaler = standardizar_dados(
    X_train,
    X_test
)

print("X_train_scaled:")
print(X_train_scaled.shape)

print("\nX_test_scaled:")
print(X_test_scaled.shape)

In [ ]:
print("--- ANTES DA STANDARDIZAÇÃO ---")
display(X_train.head())

print("\n--- DEPOIS DA STANDARDIZAÇÃO ---")
display(X_train_scaled.head())

## Tratamento dos outliers

Durante o Data Understanding foram identificados valores extremos em várias características, nomeadamente `SizeOfCode`, `SizeOfImage` e `OverlaySize`.

Nesta fase optou-se por não remover automaticamente esses valores.

Em datasets de malware, valores estruturais pouco frequentes podem representar características relevantes de ficheiros maliciosos. A remoção automática desses registos poderia eliminar informação útil para a classificação.

O impacto dos valores extremos poderá ser posteriormente avaliado durante a comparação dos modelos.

## Feature Selection

A seleção inicial de características resultou das conclusões do Data Understanding.

Foram removidas:

- `File`, por funcionar como identificador;
- `SuspiciousSectionNames`, por ser constante;
- `SectionCount`, devido à redundância com `NumberOfSections`.

Também foram observadas correlações elevadas entre algumas outras características, como `BaseOfCode` e `SizeOfUninitializedData`.

Estas variáveis não serão removidas automaticamente nesta fase. O seu impacto deverá ser avaliado posteriormente através do desempenho dos modelos.

## Feature Engineering

Nesta fase não foram criadas novas características.

O dataset já contém um conjunto significativo de propriedades estruturais extraídas de ficheiros PE, incluindo tamanhos, entropias, DLL importadas, funções importadas e informação sobre secções.

Poderão ser consideradas transformações ou novas características posteriormente, caso a avaliação dos modelos indique essa necessidade.

## Feature Extraction

Não é realizada extração adicional de características durante esta fase.

As características presentes no dataset já foram previamente extraídas dos ficheiros executáveis PE e são disponibilizadas diretamente como variáveis numéricas.

Assim, o projeto utiliza as características estruturais existentes como entrada para os modelos.

## Imbalanced Data

A variável alvo apresenta uma distribuição praticamente equilibrada entre as classes benigno e malware.

Após a preparação e divisão dos dados, o equilíbrio mantém-se nos conjuntos de treino e teste.

Por este motivo, não serão utilizadas técnicas como SMOTE, oversampling ou undersampling nesta fase.

## Conclusão

A fase de Data Preparation transformou as conclusões do Data Understanding em operações concretas de preparação dos dados.

Foram removidas características desnecessárias, constantes ou redundantes, nomeadamente `File`, `SuspiciousSectionNames` e `SectionCount`.

Após a remoção destas colunas foram identificados registos duplicados que não eram visíveis no dataset original devido à presença do identificador `File`. Não foram encontrados perfis com rótulos contraditórios e os duplicados completos foram removidos.

O dataset preparado passou de 1.156 para 1.148 registos, mantendo 23 features e a variável alvo `Malicious`.

Os dados foram divididos em conjuntos de treino e teste utilizando uma proporção de 80/20 e estratificação da variável alvo.

Foi também aplicado `StandardScaler`, ajustado exclusivamente aos dados de treino para evitar data leakage.

Não foram removidos automaticamente outliers nem aplicadas técnicas de balanceamento de classes. Também não foram criadas novas features nesta fase.

Os dados encontram-se agora preparados para a fase seguinte do projeto: Model Building.